# Klasifikasi DemogPairs Menggunakan ViT (Emosi dan Wajah) & Logistic Regression

In [1]:
import numpy as np
import utils as u
import joblib
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, ParameterGrid
from imblearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from tqdm import tqdm

joblib.parallel_backend('threading')

## Load Dataset

In [2]:
data = u.load_demogpairs()
pd.DataFrame(data)

,db_code,image_path,full_path,label,label_idx
0,CWF,able_wanamakok/002.jpg,dataset/demogpairs/images\able_wanamakok/002.jpg,Asian_Females,5
1,CWF,able_wanamakok/004.jpg,dataset/demogpairs/images\able_wanamakok/004.jpg,Asian_Females,5
2,CWF,able_wanamakok/007.jpg,dataset/demogpairs/images\able_wanamakok/007.jpg,Asian_Females,5
3,CWF,able_wanamakok/008.jpg,dataset/demogpairs/images\able_wanamakok/008.jpg,Asian_Females,5
4,CWF,able_wanamakok/012.jpg,dataset/demogpairs/images\able_wanamakok/012.jpg,Asian_Females,5
...,...,...,...,...,...
10795,CWF,zachary_quinto/177.jpg,dataset/demogpairs/images\zachary_quinto/177.jpg,White_Males,3
10796,CWF,zachary_quinto/214.jpg,dataset/demogpairs/images\zachary_quinto/214.jpg,White_Males,3
10797,CWF,zachary_quinto/217.jpg,dataset/demogpairs/images\zachary_quinto/217.jpg,White_Males,3
10798,CWF,zachary_quinto/218.jpg,dataset/demogpairs/images\zachary_quinto/218.jpg,White_Males,3


## Load Fitur

In [3]:
face_features = joblib.load('features/demogpairs_vit-face.pkl')
emotion_features = joblib.load('features/demogpairs_vit-emotion.pkl')
features = {}
for d in tqdm(data):
    key = d['image_path']
    features[key] = np.array(list(face_features[key]) + list(emotion_features[key]))
print('Jumlah fitur per gambar:', np.array(features[list(features.keys())[0]]).shape[0])

100%|█████████████████████████████████████████████████████████████████████████| 10800/10800 [00:01<00:00, 10687.46it/s]

Jumlah fitur per gambar: 1536


## Split Data

In [4]:
X = np.array([features[d['image_path']] for d in data])
y = np.array([d['label_idx'] for d in data])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print((len(X_train), len(X_test)))

(8640, 2160)


## Kombinasi Parameter

In [5]:
grid_params = [
    {
        'scaler': [None, MinMaxScaler()],
        'pca': [None, PCA(n_components=0.5), PCA(n_components=0.75)],
        
        'classifier': [LogisticRegression(random_state=42)],
        'classifier__C': [0.01, 0.1, 1, 10],
        'classifier__max_iter': [500, 1000],
        'classifier__solver': ['lbfgs', 'saga'],
    },
]

pipeline = Pipeline(steps=[
    ('scaler', None),
    ('pca', None),
    ('classifier', None)
])

skv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    'accuracy': 'accuracy', 
    'f1': 'f1_macro', 
    'precision': 'precision_macro', 
    'recall': 'recall_macro',
}

grid_models = {}
for params in grid_params:
    key = str(params['classifier'][0]).split('(')[0]
    grid_models[key] = GridSearchCV(
        estimator=pipeline,
        param_grid=params,
        cv=skv, refit='accuracy',
        scoring=scoring, n_jobs=int(joblib.cpu_count() * 0.6),
        verbose=1, error_score='raise',
        return_train_score=True
    )
    print(f'{key}: {len(ParameterGrid(params))} kombinasi')

LogisticRegression: 96 kombinasi


## Klasifikasi

In [6]:
evaluation_results, fold_results = u.evaluate_models(
    grid_models, 
    X_train, y_train,
    X_test, y_test,
    target_names=u.demogpairs_classes,
    model_prefix="models/clf_demogpairs_lr_vit-emotion-face_",
    results_path="results/demogpairs_lr_vit-emotion-face_"
)
sorted_results = pd.DataFrame(evaluation_results).sort_values(by="test_accuracy", ascending=False).to_dict("records")
u.html_br()
_dtable = u.display_table(sorted_results)

Evaluating: LogisticRegression


{'classifier': 'LogisticRegression', 'classifier__C': 0.1, 'classifier__max_iter': 500, 'classifier__solver': 'lbfgs', 'pca': None, 'scaler': None}


Accuracy  : 0.924074074074074
Precision : 0.9240818240676675
Recall    : 0.924074074074074
F1 Score  : 0.9240193965624607
               precision    recall  f1-score   support

Asian_Females     0.9466    0.9361    0.9413       360
  Asian_Males     0.9126    0.9278    0.9201       360
Black_Females     0.9071    0.9222    0.9146       360
  Black_Males     0.9479    0.9611    0.9545       360
White_Females     0.9237    0.9083    0.9160       360
  White_Males     0.9065    0.8889    0.8976       360

     accuracy                         0.9241      2160
    macro avg     0.9241    0.9241    0.9240      2160
 weighted avg     0.9241    0.9241    0.9240      2160



Class,OvR Accuracy,Precision,Recall,F1-Score,Support
Asian_Females,0.9805555555555555,0.9466292134831461,0.9361111111111111,0.941340782122905,360
Asian_Males,0.9731481481481481,0.912568306010929,0.9277777777777778,0.9201101928374655,360
Black_Females,0.9712962962962963,0.907103825136612,0.9222222222222223,0.9146005509641874,360
Black_Males,0.9847222222222223,0.947945205479452,0.9611111111111111,0.9544827586206897,360
White_Females,0.9722222222222222,0.923728813559322,0.9083333333333333,0.9159663865546218,360
White_Males,0.9662037037037037,0.9065155807365439,0.8888888888888888,0.8976157082748948,360


Confusion matrix saved: images\cm_lr_vit-emotion-face_LogisticRegression.png



Confusion Matrix:
                         Asian_Females       Asian_Males     Black_Females       Black_Males     White_Females       White_Males
       Asian_Females               337                 0                10                 5                 8                 0
         Asian_Males                 0               334                 0                 4                 8                14
       Black_Females                 7                 0               332                 8                 1                12
         Black_Males                 4                 4                 6               346                 0                 0
       White_Females                 7                15                 3                 1               327                 7
         White_Males                 1                13                15                 1                10               320


model_name,model_file_path,best_parameters,test_accuracy,test_f1,test_precision,test_recall,parameter_combinations
LogisticRegression,models/clf_demogpairs_lr_vit-emotion-face_LogisticRegression.pkl,"{'classifier': 'LogisticRegression', 'classifier__C': 0.1, 'classifier__max_iter': 500, 'classifier__solver': 'lbfgs', 'pca': None, 'scaler': None}",0.924074074074074,0.9240193965624607,0.9240818240676675,0.924074074074074,270


In [7]:
model, training_time = u.load_object('models/clf_demogpairs_lr_vit-emotion-face_LogisticRegression.pkl')
u.h(5, 'Waktu Pelatihan (Jobs)')
u.seconds_to_time(round(training_time))

{'input_seconds': 7086.0,
 'days': 0,
 'hours': 1,
 'minutes': 58,
 'seconds': 6.0,
 'text': '0 hari 1 jam 58 menit 6.0 detik'}

In [8]:
u.h(5, 'Waktu Pelatihan')
times = [fr['Train Time Mean'] * 5 for fr in fold_results]
u.seconds_to_time(round(np.sum(times) + model.refit_time_))

{'input_seconds': 80787.0,
 'days': 0,
 'hours': 22,
 'minutes': 26,
 'seconds': 27.0,
 'text': '0 hari 22 jam 26 menit 27.0 detik'}

In [9]:
_dtable = u.display_table(fold_results, n_items=[4, 4], column_widths=['5%', '45%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%'])

No,Params,Fold 1,Fold 2,Fold 3,Fold 4,Fold 5,Accuracy Mean,F1 Score Mean,Precision Mean,Recall Mean,Train Time Mean
1,"{'classifier': 'LogisticRegression', 'classifier__C': 0.1, 'classifier__max_iter': 500, 'classifier__solver': 'lbfgs', 'pca': None, 'scaler': None}",0.9259,0.9201,0.9109,0.9184,0.919,0.9189,0.9189,0.9192,0.9189,22.9332
2,"{'classifier': 'LogisticRegression', 'classifier__C': 0.1, 'classifier__max_iter': 2000, 'classifier__solver': 'lbfgs', 'pca': None, 'scaler': None}",0.9259,0.9201,0.9109,0.9184,0.919,0.9189,0.9189,0.9192,0.9189,20.0105
3,"{'classifier': 'LogisticRegression', 'classifier__C': 0.1, 'classifier__max_iter': 1000, 'classifier__solver': 'lbfgs', 'pca': None, 'scaler': None}",0.9259,0.9201,0.9109,0.9184,0.919,0.9189,0.9189,0.9192,0.9189,18.6106
4,"{'classifier': 'LogisticRegression', 'classifier__C': 1, 'classifier__max_iter': 500, 'classifier__solver': 'lbfgs', 'pca': None, 'scaler': 'MinMaxScaler'}",0.9236,0.919,0.9138,0.9167,0.9161,0.9178,0.9179,0.9182,0.9178,81.8854
...,...,...,...,...,...,...,...,...,...,...,...
267,"{'classifier': 'LogisticRegression', 'classifier__C': 0.01, 'classifier__max_iter': 1000, 'classifier__solver': 'saga', 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.8773,0.8565,0.8576,0.8704,0.8617,0.8647,0.8642,0.8652,0.8647,18.6994
268,"{'classifier': 'LogisticRegression', 'classifier__C': 0.01, 'classifier__max_iter': 2000, 'classifier__solver': 'newton-cg', 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.8773,0.8565,0.8576,0.8704,0.8617,0.8647,0.8642,0.8652,0.8647,11.9102
269,"{'classifier': 'LogisticRegression', 'classifier__C': 0.01, 'classifier__max_iter': 2000, 'classifier__solver': 'saga', 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.8773,0.8565,0.8576,0.8704,0.8617,0.8647,0.8642,0.8652,0.8647,20.4316
270,"{'classifier': 'LogisticRegression', 'classifier__C': 0.01, 'classifier__max_iter': 1000, 'classifier__solver': 'newton-cg', 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.8773,0.8565,0.8576,0.8704,0.8617,0.8647,0.8642,0.8652,0.8647,13.8036
